# Chandraghanta: is this network flow an attack?

CIC-IDS2017, attached from its public Kaggle copy (it is not redistributed here). The usual protocol splits flows at random and reports near-perfect scores. Here the split is **by day**: train on Monday–Wednesday, test on Thursday–Friday, so every test attack family is one the model never saw.

Chandraghanta is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
frames = []
for path in sorted(glob.glob("/kaggle/input/**/*-no-metadata.parquet", recursive=True)):
    df = pd.read_parquet(path)
    df["day"] = os.path.basename(path).split("-")[1]
    frames.append(df)
data = pd.concat(frames, ignore_index=True)
label = next(c for c in data.columns if c.strip().lower() == "label")
data["attack"] = (data[label].astype(str).str.upper() != "BENIGN").astype(int)
print(data.shape); data.groupby("day")["attack"].mean().round(3)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score
features = [c for c in data.columns if c not in (label, "day", "attack") and pd.api.types.is_numeric_dtype(data[c])]
X = data[features].replace([np.inf, -np.inf], np.nan)
train = data["day"].isin(["Monday", "Tuesday", "Wednesday"])
model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, verbose=-1)
model.fit(X[train], data.loc[train, "attack"])
pred = model.predict(X[~train])
print(f"F1 on unseen days: {f1_score(data.loc[~train, 'attack'], pred):.3f}")

from sklearn.model_selection import train_test_split
xa, xb, ya, yb = train_test_split(X, data["attack"], test_size=0.2, random_state=7)
print(f"F1 on a random split: {f1_score(yb, lgb.LGBMClassifier(n_estimators=300, verbose=-1).fit(xa, ya).predict(xb)):.3f}")

In [ ]:
test = data[~train].assign(pred=pred)
test[test["attack"] == 1].groupby(label)["pred"].agg(["mean", "size"]).rename(columns={"mean": "recall", "size": "flows"}).round(3)

**Reading it:** the gap between the two F1 scores is the point. A random split puts flows from the same attack session on both sides, so it measures recognition of known traffic, not detection of new attacks. Per family, brute-force-like web attacks transfer; port scans and bots do not. The attached results file holds Chandraghanta's own numbers (F1 0.997 random, 0.453 unseen days).